<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab09.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 9 — VQE for Correlated Electrons: the Hubbard Dimer

**Maps to:** Module 4 (the whole arc), applied to a materials model

**Time:** ~60–75 minutes (instructor walkthrough ~15 min). Total compute ~2 minutes.

---

### The model

Two atomic sites. Electrons can **hop** between them (amplitude $t$), and pay an energy
penalty $U$ whenever two electrons sit on the same site:

$$ H = -t\sum_{\sigma}\big(c^\dagger_{1\sigma}c_{2\sigma} + \text{h.c.}\big)
\;+\; U\sum_{i} n_{i\uparrow}n_{i\downarrow} .$$

This is the smallest model of **correlated electrons in a solid**. The ratio $U/t$ is the
single most important number in the electronic structure of transition metals, catalytic
surfaces, and the conductive/insulating behaviour of oxide coatings:

* $U/t \ll 1$ — electrons delocalize; **metallic**, and mean-field theory works.
* $U/t \gg 1$ — electrons localize one per site to avoid each other; a **Mott insulator**,
  where mean-field theory fails badly no matter how you tune it.

### The punchline you should watch for

The Hubbard dimer at half filling **is** minimal-basis H$_2$ in disguise. "Bonding vs
antibonding" becomes "delocalized vs localized"; the "tiny percentage of antibonding
configuration" from Module 3 becomes the correlation that destroys the metallic state.
Same mathematics, same circuit, different vocabulary — which is exactly why one algorithm
serves chemistry and materials science at once.

### After this lab you can
1. Hand-build a fermionic lattice Hamiltonian in Pauli form via Jordan–Wigner.
2. Use a **symmetry-preserving** ansatz and explain why it beats a generic one here.
3. Reproduce the analytic ground-state energy across the whole $U/t$ range.
4. Measure double occupancy and show quantitatively where mean-field theory breaks.
5. Adapt the notebook to a Hamiltonian of your own.

In [ ]:
# %pip install -q qiskit qiskit-aer matplotlib scipy
import numpy as np, time
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import excitation_preserving, efficient_su2
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2

np.set_printoptions(precision=4, suppress=True)
sim = AerSimulator()
estimator = EstimatorV2(options={"default_precision": 0.0})
print("ready")

## 1. Four spin orbitals again

Two sites × two spins = four spin orbitals = four qubits, with $|1\rangle$ = occupied:

| qubit | spin orbital |
|---|---|
| 0 | site 1, ↑ |
| 1 | site 2, ↑ |
| 2 | site 1, ↓ |
| 3 | site 2, ↓ |

Under Jordan–Wigner, the two pieces of the Hamiltonian become:

* **Hopping** between *adjacent* orbital indices needs no $Z$ string (Lab 6):
  $$c_0^\dagger c_1 + c_1^\dagger c_0 \;=\; \tfrac12\,(X_0X_1 + Y_0Y_1).$$
  That is the Module 3 mixer, showing up as the physics of electron delocalization.
* **Number operators**: $n_p = \tfrac12(I - Z_p)$, so the interaction
  $U\,n_0 n_2$ expands to four terms.

In [ ]:
def plabel(n, ops):
    s = ["I"] * n
    for q, p in ops.items():
        s[q] = p
    return "".join(s[::-1])

def hubbard_dimer(t=1.0, U=0.0):
    '''2-site Hubbard model at half filling, Jordan-Wigner encoded on 4 qubits.'''
    n, terms = 4, []
    for a, b in [(0, 1), (2, 3)]:                    # hopping, per spin channel
        terms += [(plabel(n, {a: "X", b: "X"}), -t/2),
                  (plabel(n, {a: "Y", b: "Y"}), -t/2)]
    for a, b in [(0, 2), (1, 3)]:                    # U * n_up n_down, per site
        terms += [(plabel(n, {}), U/4), (plabel(n, {a: "Z"}), -U/4),
                  (plabel(n, {b: "Z"}), -U/4), (plabel(n, {a: "Z", b: "Z"}), U/4)]
    return SparsePauliOp.from_list(terms).simplify()

def number_operator(n=4):
    return sum(SparsePauliOp.from_list([(plabel(n, {}), 0.5),
                                        (plabel(n, {p: "Z"}), -0.5)])
               for p in range(n)).simplify()

print(hubbard_dimer(t=1.0, U=4.0))

### Exercise 1 — check against the analytic answer

The half-filled dimer (2 electrons, one of each spin) has a closed-form ground state:

$$ E_0 = \frac{U - \sqrt{U^2 + 16t^2}}{2}. $$

Two limits worth checking by hand: at $U=0$, $E_0 = -2t$ (both electrons in the bonding
orbital); as $U\to\infty$, $E_0 \to -4t^2/U \to 0$ (electrons frozen one per site).

In [ ]:
def analytic_E0(t, U):
    # TODO: (U - sqrt(U^2 + 16 t^2)) / 2
    ...

N_op = number_operator()

def exact_half_filled(t, U):
    '''Lowest eigenvalue restricted to the 2-electron sector.'''
    w, v = np.linalg.eigh(hubbard_dimer(t, U).to_matrix())
    Nm = N_op.to_matrix()
    best = np.inf
    for k in range(len(w)):
        n_e = np.real(v[:, k].conj() @ Nm @ v[:, k])
        if abs(n_e - 2) < 1e-6:
            best = min(best, w[k])
    return best

print(f"{'U/t':>6}{'exact (N=2)':>14}{'analytic':>12}{'global min':>13}")
for U in [0, 1, 2, 4, 8]:
    ex = exact_half_filled(1.0, U)
    gl = np.linalg.eigvalsh(hubbard_dimer(1.0, U).to_matrix())[0]
    print(f"{U:>6}{ex:>14.6f}{analytic_E0(1.0, U):>12.6f}{gl:>13.6f}")
    assert np.isclose(ex, analytic_E0(1.0, U))
print("\nPASS -- but look at the last column.")

## 2. A trap worth falling into once

For $U \gtrsim 4t$ the **global** minimum of that Hamiltonian is not the state we want:
it lies in a sector with the wrong number of electrons. Our Hamiltonian says nothing
about how many electrons the system has — nothing forbids the optimizer from emptying the
molecule to avoid the repulsion.

VQE minimizes energy. It will find that unphysical state and report it, with no error
message. **Symmetry is not automatic — you have to impose it.**

Two standard fixes:

1. **Penalty term:** minimize $H + \mu(\hat N - 2)^2$. Simple, always available, but adds
   Pauli terms (more measurements) and needs $\mu$ tuned.
2. **Symmetry-preserving ansatz:** use a circuit that cannot change the particle number,
   started from a state with the right number. Free at runtime, and strictly better when
   available. `excitation_preserving` builds exactly this — its two-qubit blocks are the
   $\mathrm{R}_{XX}\mathrm{R}_{YY}$ mixer from Lab 3, which only ever moves $|01\rangle
   \leftrightarrow |10\rangle$.

### Exercise 2 — build both and watch the first one fail

In [ ]:
def hf_state():
    '''One electron of each spin on site 1: qubits 0 and 2 occupied.'''
    qc = QuantumCircuit(4)
    qc.x(0); qc.x(2)
    return qc

def preserving_ansatz(reps=2):
    qc = hf_state().compose(excitation_preserving(4, reps=reps, entanglement="full"))
    return transpile(qc, sim, optimization_level=1)

def generic_ansatz(reps=2):
    return transpile(efficient_su2(4, reps=reps, entanglement="linear"),
                     sim, optimization_level=1)

anz_keep = preserving_ansatz()
anz_free = generic_ansatz()
print("excitation-preserving:", anz_keep.num_parameters, "params, depth", anz_keep.depth())
print("generic efficient_su2:", anz_free.num_parameters, "params, depth", anz_free.depth())

print("\nHF reference particle number:",
      float(estimator.run([(hf_state(), N_op)]).result()[0].data.evs))

In [ ]:
def vqe(H, ansatz, n_restarts=3, maxiter=500, seed0=0, spread=0.5):
    best_e, best_x = np.inf, None
    f = lambda x: float(estimator.run([(ansatz, H, x)]).result()[0].data.evs)
    for s in range(n_restarts):
        rng = np.random.default_rng(seed0 + s)
        x0 = rng.uniform(-spread, spread, ansatz.num_parameters)
        res = minimize(f, x0, method="COBYLA", options={"maxiter": maxiter})
        if res.fun < best_e:
            best_e, best_x = res.fun, res.x
    return best_e, best_x

U_demo = 8.0
H_demo = hubbard_dimer(1.0, U_demo)

e_free, x_free = vqe(H_demo, anz_free, n_restarts=3)
n_free = float(estimator.run([(anz_free, N_op, x_free)]).result()[0].data.evs)

e_keep, x_keep = vqe(H_demo, anz_keep, n_restarts=3)
n_keep = float(estimator.run([(anz_keep, N_op, x_keep)]).result()[0].data.evs)

print(f"target (half filled, analytic): {analytic_E0(1.0, U_demo):+.5f}   N = 2")
print(f"generic ansatz               : {e_free:+.5f}   N = {n_free:.3f}")
print(f"excitation-preserving ansatz : {e_keep:+.5f}   N = {n_keep:.3f}")
print("\nThe generic ansatz reports a LOWER energy -- by describing a different,")
print("unphysical system. A number that beats the reference is not always good news.")
assert abs(n_keep - 2) < 1e-6

## 3. The metal-to-insulator crossover

Now sweep $U/t$ with the symmetry-preserving ansatz and track:

* the ground-state energy against the analytic curve;
* the **double occupancy** $D = \langle n_{1\uparrow}n_{1\downarrow}\rangle$ — the
  probability of finding both electrons on the same site. This is the direct measure of
  how strongly the electrons are correlated. $D = 0.25$ means they ignore each other
  (uncorrelated); $D \to 0$ means each has claimed its own site.

Runtime: ~70 seconds.

In [ ]:
D_op = SparsePauliOp.from_list([(plabel(4, {}), 0.25), (plabel(4, {0: "Z"}), -0.25),
                                (plabel(4, {2: "Z"}), -0.25),
                                (plabel(4, {0: "Z", 2: "Z"}), 0.25)]).simplify()

t0 = time.time()
U_values = np.array([0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 6.0, 8.0])
E_vqe, E_ana, docc = [], [], []

for U in U_values:
    Hu = hubbard_dimer(1.0, U)
    e, x = vqe(Hu, anz_keep, n_restarts=3, maxiter=600)
    d = float(estimator.run([(anz_keep, D_op, x)]).result()[0].data.evs)
    E_vqe.append(e); E_ana.append(analytic_E0(1.0, U)); docc.append(d)
    print(f"U/t={U:4.1f}  E_vqe={e:+9.5f}  analytic={E_ana[-1]:+9.5f}  "
          f"err={1000*(e-E_ana[-1]):7.3f} mHa-equiv   double occupancy={d:.4f}")

print(f"\ntotal {time.time()-t0:.1f} s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(U_values, E_ana, "k-", lw=2, label="analytic")
ax[0].plot(U_values, E_vqe, "o", ms=6, label="VQE")
ax[0].set_xlabel("U / t"); ax[0].set_ylabel("ground state energy / t")
ax[0].set_title("Hubbard dimer, half filling"); ax[0].legend(fontsize=8)

ax[1].plot(U_values, docc, "s-", label="VQE double occupancy")
ax[1].axhline(0.25, color="r", ls=":", label="uncorrelated (mean field)")
ax[1].set_xlabel("U / t"); ax[1].set_ylabel(r"$\langle n_{1\uparrow}n_{1\downarrow}\rangle$")
ax[1].set_title("Electrons learning to avoid each other"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Where mean-field theory breaks

Mean-field (Hartree–Fock) treats each electron as moving in the *average* field of the
others — your Module 4 slide showed this as a car responding to smoothed-out traffic
density rather than to individual cars. For this model that approximation is exactly
$D = 0.25$, independent of $U$: the electrons never learn to dodge.

### Exercise 3 — quantify the failure

The uncorrelated (Hartree–Fock) energy of the half-filled dimer is $E_{HF} = -2t + U/2$.
Compare it with the true energy.

In [ ]:
E_hf = ...          # TODO: array of -2 + U/2 over U_values
E_corr = np.array(E_vqe) - E_hf

print(f"{'U/t':>6}{'E_HF':>10}{'E_exact':>10}{'correlation energy':>21}{'% of |E|':>10}")
for U, ehf, e in zip(U_values, E_hf, E_vqe):
    pct = 100*abs(e - ehf)/max(abs(e), 1e-9)
    print(f"{U:>6.1f}{ehf:>10.4f}{e:>10.4f}{e-ehf:>21.4f}{pct:>10.1f}")

### Exercise 4 — close the loop with Lab 5

In Lab 5 you plotted the antibonding weight of H$_2$ against bond length $R$ and watched
it climb from ~1% to ~40% as the bond stretched. Plot the analogous quantity here — how
far the ground state is from the uncorrelated one — against $U/t$, and put the two stories
side by side.

In [ ]:
overlap_hf = []
for U in U_values:
    _, x = vqe(hubbard_dimer(1.0, U), anz_keep, n_restarts=3, maxiter=600)
    sv = Statevector(anz_keep.assign_parameters(x))
    p = sv.probabilities_dict()
    ionic = p.get("0101", 0.0) + p.get("1010", 0.0)     # both electrons on one site
    overlap_hf.append(ionic)

plt.figure(figsize=(6.2, 3.4))
plt.plot(U_values, overlap_hf, "o-")
plt.axhline(0.5, color="r", ls=":", label="uncorrelated limit")
plt.xlabel("U / t"); plt.ylabel("weight of doubly-occupied configurations")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

print("Compare with Lab 5: stretching an H2 bond and turning up U in a metal are")
print("the same phenomenon. In both cases a single configuration stops being enough,")
print("and the ground state becomes an even mix -- which is precisely the regime")
print("where classical mean-field methods fail and a wavefunction method is required.")

## 5. Bring your own Hamiltonian

Everything above is a template. To study a different model you change **one function** —
the Hamiltonian — and possibly the ansatz. The loop, the optimizer, the measurement, and
the hardware pipeline are untouched.

### Exercise 5 (open-ended) — pick one and run it

**(a) Heisenberg dimer.** In the large-$U$ limit the Hubbard model reduces to a spin model
with exchange $J = 4t^2/U$:
$$H = J\,(X_0X_1 + Y_0Y_1 + Z_0Z_1)/4 .$$
Build it on 2 qubits, find the ground state, and check $E_0 = -3J/4$ (the singlet).
Then verify the $4t^2/U$ prediction against your Hubbard energies at large $U$.

**(b) Three-site chain.** Extend `hubbard_dimer` to 3 sites (6 qubits). Watch the Pauli
term count and the ansatz depth grow. Compare VQE against exact diagonalization while you
still can.

**(c) Ising with a longitudinal field.** Add $-g\sum_i Z_i$ to Lab 8's TFIM — the model
for a magnetic material in a biasing field. Map the phase diagram in $(h, g)$.

**(d) Your own.** Any Hamiltonian you can write as a weighted sum of Pauli strings works.

A skeleton is provided below.

In [ ]:
def my_hamiltonian(**params):
    '''Replace with your model. Return a SparsePauliOp.'''
    J = params.get("J", 1.0)
    return SparsePauliOp.from_list([("XX", J/4), ("YY", J/4), ("ZZ", J/4)])

def study(H, ansatz, label=""):
    e, x = vqe(H, ansatz, n_restarts=4, maxiter=600)
    e_exact = np.linalg.eigvalsh(H.to_matrix())[0]
    print(f"{label:20s} VQE {e:+.6f}   exact {e_exact:+.6f}   error {e-e_exact:.2e}")
    return e, x

J = 1.0
H_heis = my_hamiltonian(J=J)
anz2 = transpile(excitation_preserving(2, reps=2, entanglement="linear"),
                 sim, optimization_level=1)
init2 = QuantumCircuit(2); init2.x(0)
anz2 = transpile(init2.compose(excitation_preserving(2, reps=2)), sim, optimization_level=1)

e, _ = study(H_heis, anz2, "Heisenberg dimer")
print(f"singlet prediction   -3J/4 = {-3*J/4:+.6f}")

## 6. Where this leaves you

Over nine labs you built, from the bottom up:

| you built | in labs |
|---|---|
| complex amplitudes and probabilities | 1 |
| Pauli operators and the exponential-of-a-generator trick | 2 |
| the mixer that superposes configurations | 3 |
| energy estimation from raw bitstrings | 4 |
| the complete VQE loop for a molecule | 5 |
| fermionic encodings and chemistry-derived ansätze | 6 |
| the same circuit on real hardware, with mitigation | 7 |
| a materials model with a phase transition | 8 |
| correlated electrons, and the limits of mean-field theory | 9 |

What is honestly true today: for H$_2$ this is a demonstration, not an advantage — your
laptop diagonalized every Hamiltonian in this course faster than the VQE loop ran. The
open question is where the crossover is, and whether error rates fall fast enough to
reach it before fault-tolerant methods arrive and make VQE obsolete. Both outcomes are
live. What is not in doubt is that the ingredients — Hamiltonians as Pauli sums, ansatz
design, measurement grouping, error mitigation — are the vocabulary of the field either
way.

## Checkpoint

1. Why does the Hubbard hopping term produce exactly the $X_0X_1 + Y_0Y_1$ operator from
   Module 3?
2. Your VQE reports an energy below the analytic ground state. Give the physical
   explanation (not "it is a bug").
3. Name the two ways to enforce particle-number conservation and one drawback of each.
4. Double occupancy falls from 0.25 toward 0 as $U/t$ grows. Explain what a Hartree–Fock
   calculation would predict, and why.
5. Which is harder for VQE — H$_2$ at $R = 3$ Å, or the Hubbard dimer at $U/t = 8$?
   Argue in terms of the ground state, not the qubit count.